# Generate a golden dataset with an LLM, review it, train on it

**Needs an LLM:** set `ANTHROPIC_API_KEY` (or `OPENAI_API_KEY`, `GEMINI_API_KEY`, ...), or run Ollama locally and
set `DS_TEACHER=ollama/qwen3`. This notebook is committed without outputs because running it calls a paid API.
CI runs it with `DS_OFFLINE=1`, where a local stand-in writes placeholder texts, only to check that the code runs.

`model.generate(n, teacher=...)` gives every label an equal share, asks for a different style in each batch, drops
duplicates, then has the teacher label every text again blind and keeps only the ones it labels the same way.
Held-out numbers on LLM-written data are optimistic, so the last step tests on texts you wrote yourself.

Runs on: a laptop (CPU or MPS) plus an LLM key or a local Ollama model.

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations. decisionsmith is an independent project, not affiliated with TypeSafe AI or Convai Innovations.

In [ ]:
# uv pip install "decisionsmith[laya,anthropic]"
import os

import decisionsmith as ds

TEACHER = os.environ.get("DS_TEACHER", "claude-haiku-4-5")

In [ ]:
model = ds.model(["billing", "technical", "sales"], question="Which team should handle this ticket?")
rows = model.generate(120, teacher=TEACHER, about="customer support emails for a SaaS product")
print(len(rows), "rows kept")

## Review
Write them to a CSV and read it. Delete or fix rows you disagree with before training.

`model.generate` writes synthetic texts. To build a golden set from real traffic instead, use `ds.golden("decisions.db", teacher=..., schema=...)`: it picks the logged texts most worth labelling, has the LLM label them and writes `golden.csv` with a `split` column.

In [ ]:
import csv

with open("golden.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["text", "label"])
    w.writerows([r["text"], r["answers"]["label"]] for r in rows)
print(open("golden.csv", encoding="utf-8").read()[:1500])

In [ ]:
report = model.train("golden.csv", out="runs/golden")
print(report)

## Test on texts you wrote
These never went through the LLM.

In [ ]:
mine = [
    ("You billed my card twice for March.", "billing"),
    ("The export button does nothing on Firefox.", "technical"),
    ("Do you do discounts for schools?", "sales"),
    ("Where can I download my receipts?", "billing"),
    ("I get a 404 after logging in.", "technical"),
    ("Can we move to yearly billing for 30 seats?", "sales"),
]
preds = model.predict([t for t, _ in mine])
print("%d of %d right" % (sum(p == y for p, (_, y) in zip(preds, mine)), len(mine)))
for (text, label), pred in zip(mine, preds):
    print("%-9s (want %-9s) %s" % (pred, label, text))